In [1]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              # Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Cell 3 — Verify uploaded inputs
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
ASEAN-Guide-on-AI-Governance-and-Ethics_beautified_201223_v2.pdf bytes: 4233765
NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf bytes: 9772377
Upload verified. Select your relevant input(s) for Step A3.


In [5]:
# Cell A3 — Inspect two representative PDFs
!pip -q install pypdf
from pypdf import PdfReader

SOURCES = {
    "S01": "NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf",
    "S02": "ASEAN-Guide-on-AI-Governance-and-Ethics_beautified_201223_v2.pdf",
}

for sid, name in SOURCES.items():
    p = RAW_DIR / name
    assert p.is_file(), f"{sid} not found: {p}"
    reader = PdfReader(str(p))
    pages = [(pg.extract_text() or "") for pg in reader.pages]
    empty = sum(1 for t in pages if not t.strip())
    mid = len(pages) // 2
    print(f"\n=== {sid}: {name}")
    print("Size (bytes):", p.stat().st_size)
    print("Pages:", len(pages), "| pages with no text:", empty)
    print("Total characters:", sum(len(t) for t in pages))
    print("Page 1 sample:", repr(pages[0][:250]))
    print(f"Page {mid + 1} sample:", repr(pages[mid][:400]))


=== S01: NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf
Size (bytes): 9772377
Pages: 138 | pages with no text: 2
Total characters: 256364
Page 1 sample: '“AI FOR MALAYSIA, AI FOR ALL”\nTo Enhance The Development and Deployment of AI Technology\nTHE NATIONAL GUIDELINES\nON AI GOVERNANCE & \nETHICS\n'
Page 70 sample: 'PART B\nB3.0 WHAT YOU COULD DO AS POLICY MAKER\n3.1 THE AI GOVERNANCE AND ETHICS (AIGE) FRAMEWORK FOR AI \nSYSTEMS\nEthical debates often predate legal regulatory initiatives, and the\nlifecycle of AI is no exception. Where the risks and challenges arising\nfrom the design and use of AI require managing, in that case, ethical\nframeworks can be introduced in place of or before legal regulation.\nEthical f'

=== S02: ASEAN-Guide-on-AI-Governance-and-Ethics_beautified_201223_v2.pdf
Size (bytes): 4233765
Pages: 87 | pages with no text: 0
Total characters: 191310
Page 1 sample: 'ASEAN Guide on\nAI Governance and Ethics\n'
Page 44 sample: 'GOJEK\nIllustration on stakeholder interacti

In [6]:
#a4 data inventory
from pypdf import PdfReader
import re

r1 = PdfReader(str(RAW_DIR / "NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf"))
print("S01 empty pages:", [i + 1 for i, p in enumerate(r1.pages) if not (p.extract_text() or "").strip()])

r2 = PdfReader(str(RAW_DIR / "ASEAN-Guide-on-AI-Governance-and-Ethics_beautified_201223_v2.pdf"))
t2 = "\n".join(p.extract_text() or "" for p in r2.pages)
print("S02 'AI':", len(re.findall(r"\bAI\b", t2)), "| 'Al':", len(re.findall(r"\bAl\b", t2)))

S01 empty pages: [3, 138]
S02 'AI': 1053 | 'Al': 42


In [7]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
 if relative_path:
  selected = RAW_DIR / relative_path
  if not selected.is_file():
    raise FileNotFoundError(
      f"Source not found: {selected}"
    )
  return selected
 matches = sorted(
  p for p in RAW_DIR.rglob("*")
  if p.is_file()
  and (filename is None or p.name == filename)
  and (suffixes is None or p.suffix.lower() in suffixes)
 )
 print(
 "Candidates:",
 [str(p.relative_to(RAW_DIR)) for p in matches]
 )
 if len(matches) != 1:
  raise ValueError(
    "Select exactly one source. "
    "Set relative_path to an exact file under RAW_DIR."
 )
  return matches[0]
def source_hash(path):
  return hashlib.sha256(
    Path(path).read_bytes()
  ).hexdigest()
def save_manifest(records):
  manifest = {
 "pipeline_version": "week3-v1",
 "generated_at_utc": datetime.now(timezone.utc).isoformat(),
 "records": records

  }
  target = BASE_DIR / "data" / "week3_manifest.json"
  target.parent.mkdir(parents=True, exist_ok=True)
  target.write_text(
    json.dumps(
      manifest,
      indent=2,
      ensure_ascii=False
    ),
      encoding="utf-8"
  )
  print("Saved lineage:", target)
  return manifest
# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [8]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )

    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )

    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )

    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [9]:
# Cell 5A1 — PATHWAY A1 — PDF/TXT
!pip -q install pypdf
from pypdf import PdfReader
import re

f = select_source(relative_path="NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf")

if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")

text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))
assert clean, "No usable text: check extraction or OCR needs."

def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts, current = [], ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))
print("\nFixed sample:", repr(fixed[0][:180]))
print("\nParagraph-aware sample:", repr(structured[0][:180]))
print("\nPATHWAY A1 completed successfully.")

Source: NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf
Pages/segments: 138
Extracted characters: 256501
Cleaned characters: 256438
Content sample: '“AI FOR MALAYSIA, AI FOR ALL”\nTo Enhance The Development and Deployment of AI Technology\nTHE NATIONAL GUIDELINES\nON AI GOVERNANCE & \nETHICS\n\nPublished by: \nMinistry of Science, Technology and Innovation\nFloors 1-7, Block C4 & C5, Complex C, \nFederal Government Administrative Centre \n62662 Putrajaya,'

--- Chunking Comparison ---
Fixed chunks: 513
Paragraph-aware chunks: 515

Fixed sample: '“AI FOR MALAYSIA, AI FOR ALL”\nTo Enhance The Development and Deployment of AI Technology\nTHE NATIONAL GUIDELINES\nON AI GOVERNANCE & \nETHICS\n\nPublished by: \nMinistry of Science, Tec'

Paragraph-aware sample: '“AI FOR MALAYSIA, AI FOR ALL”\nTo Enhance The Development and Deployment of AI Technology\nTHE NATIONAL GUIDELINES\nON AI GOVERNANCE & \nETHICS'

PATHWAY A1 completed successfully.


In [10]:
def stats(chunks):
    n = len(chunks)
    avg = round(sum(len(c) for c in chunks) / n)
    starts_lower = sum(1 for c in chunks if c.lstrip()[:1].islower())
    ends_no_punct = sum(1 for c in chunks if not c.rstrip().endswith((".", "!", "?", ":", ";")))
    return n, avg, starts_lower, ends_no_punct

for name, ch in [("fixed-500", fixed), ("paragraph-aware-500", structured)]:
    n, avg, sl, ep = stats(ch)
    print(f"{name}: chunks={n} | avg_len={avg} | start mid-sentence={sl} ({sl/n:.0%}) | end without punctuation={ep} ({ep/n:.0%})")

print("\nBlank-line breaks in cleaned text:", len(re.findall(r"\n\s*\n", clean)))
print("Single newlines in cleaned text:", clean.count("\n"))

import random
random.seed(1)
i = random.randrange(len(structured))
print("\nRandom paragraph-aware chunk:\n", structured[i])
print("\nRandom fixed chunk:\n", fixed[i])

fixed-500: chunks=513 | avg_len=500 | start mid-sentence=435 (85%) | end without punctuation=504 (98%)
paragraph-aware-500: chunks=515 | avg_len=498 | start mid-sentence=437 (85%) | end without punctuation=509 (99%)

Blank-line breaks in cleaned text: 5
Single newlines in cleaned text: 4999

Random paragraph-aware chunk:
 inability, interpretability, or other acts of communication and disclosure. There will be both convergence and divergence in the definition of
these principles as these National Guidelines continue to evolve alongside other standards being developed for regional and global adoption.
This National Guidelines will be voluntarily adopted by all users. To ensure comprehensive and consistent application, the definition of each seven AI
Principles in Malaysia as explained above (section 2.7.3) is part

Random fixed chunk:
 t benefits
communities, while at the same time respecting the rights and dignity of all end users. To ensure that the adoption of AI technology will not 

In [11]:
# Cell 5A2 — Choose and export traceable text chunks
import json

CHOSEN_METHOD = "fixed"

if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")

records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"

with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"

print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: fixed-char-500
Total chunks exported: 513
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 393657 bytes
Validation: PASS
First record: {'record_id': 'NATIONAL-GUIDELINES-OF-AIGE-20241118-0001', 'text': '“AI FOR MALAYSIA, AI FOR ALL”\nTo Enhance The Development and Deployment of AI Technology\nTHE NATIONAL GUIDELINES\nON AI GOVERNANCE & \nETHICS\n\nPublished by: \nMinistry of Science, Technology and Innovation\nFloors 1-7, Block C4 & C5, Complex C, \nFederal Government Administrative Centre \n62662 Putrajaya, MALAYSIA\n603 8885 8084 | enquiry@mosti.gov.my | mosti.gov.my\n©Publisher’s Copyright \nAll rights reserved. No part of this publication may be reproduced, copied, stored in any retrieval system or tran', 'source': 'NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf', 'source_sha256': '53ad6738d33e80f2692319a8e48dbc170e54b363d4104680ba37c24c5ae20a93', 'processing': 'fixed-char-500', 'page': None}


In [12]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 393657
Validation: file exists and is non-empty; verify content manually too.


In [15]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# Register the Pathway A2 results first
register_lineage(
    source=RAW_DIR / "NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf",
    output=PROCESSED_DIR / "prepared_text.jsonl",
    processing=approach
)

assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

Registered: NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf -> prepared_text.jsonl
Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/week3_project/data/week3_manifest.json


In [16]:

# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json


In [17]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)

    # INGEST
    inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    # INSPECT
    inspection = inspect_fn(inputs, configuration)

    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    # VALIDATE
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }

    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified — check before sharing",
    }

    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [18]:
# Cell 8A — CONDITIONAL REQUIRED: PDF / TXT Projects
!pip -q install pypdf
import re
from pypdf import PdfReader

SOURCE_A = resolve_source("NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf")

# Same configuration chosen in Part B (fixed-500)
CONFIG_A = {"chunk_size": 500, "method": "fixed"}

def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [page.extract_text() or "" for page in reader.pages]
        text = "\n".join(pages)
        page_count = len(pages)
    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None
    else:
        raise ValueError("Expected a PDF or TXT source")
    if not text.strip():
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")
    return {"source": src.name, "characters": len(text), "pages": page_count}

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
    else:
        raise ValueError("Expected a PDF or TXT source")

    # Same cleaning as Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    if method != "fixed":
        raise ValueError("This pipeline uses the Part B choice: 'fixed'")

    chunks = [text[i:i + size] for i in range(0, len(text), size)]
    if not chunks:
        raise ValueError("No text chunks produced")

    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {"id": i, "source": str(src), "text": chunk, "page": None}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    mapping = [{"source": str(src), "output": str(target)}]
    processing_method = f"{method} text chunks of {size} characters"
    return [target], mapping, processing_method

run_A = register_and_run("documents", SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


In [19]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")

for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")

print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [20]:
import json
p = PIPE_DIR / "documents" / "prepared_text.jsonl"
rows = [json.loads(l) for l in p.open(encoding="utf-8")]
print("Records:", len(rows), "| expected 513 (same as Part B)?", len(rows) == 513)
print("Record 50:", rows[49])

Records: 513 | expected 513 (same as Part B)? True
Record 50: {'id': 50, 'source': '/content/week3_project/data/raw/NATIONAL-GUIDELINES-OF-AIGE-20241118.pdf', 'text': 'n of its output is crucial.\n10\nImage by Freepik\nBusiness, Banking and Finance\nSectors\nExamples of Real-world AI and\nGenAI in Malaysia\n• AI-driven chatbots are used in customer service, provide instant\nresponses to inquiries, and improve the overall banking experience.\n• CIMB Bank was the first bank in Malaysia to launch an AI-driven\nchatbot, namely Enhanced Virtual Assistant (EVA).\n• HSBC Bank launched a similar Chatbot Amy for its corporate banking\nclients. Amy understands both English and simp', 'page': None}
